In [1]:
import os
from glob import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from skimage import measure
from sklearn.neighbors import NearestNeighbors

from meshtosdf import mesh_to_sdf_npz

import plotly.graph_objects as go



In [2]:
def convert_all_meshes_to_sdf(
    mesh_dir="meshes",
    out_dir="sdfs_npz",
    resolution=128,
    padding=5,
    exts=("obj", "ply", "stl")
):
    """
    Convert all mesh files in mesh_dir to SDF .npz files using mesh_to_sdf_npz.

    Returns:
      npz_paths: list of output npz paths
    """
    os.makedirs(out_dir, exist_ok=True)

    mesh_paths = []
    for ext in exts:
        mesh_paths += glob(os.path.join(mesh_dir, f"*.{ext}"))

    if len(mesh_paths) == 0:
        raise FileNotFoundError(f"No mesh files found in '{mesh_dir}' with extensions {exts}")

    npz_paths = []
    for mesh_path in mesh_paths:
        name = os.path.splitext(os.path.basename(mesh_path))[0]
        out_path = os.path.join(out_dir, f"{name}_sdf_{resolution}.npz")
        mesh_to_sdf_npz(mesh_path, out_path, resolution=resolution, padding=padding)
        npz_paths.append(out_path)

    return npz_paths


In [3]:
def pad_crop_to_shape_3d(vol: np.ndarray, target_shape=(128,128,128), pad_value=None):
    """
    Center pad/crop a 3D volume to target shape.
    """
    assert vol.ndim == 3
    D, H, W = vol.shape
    Dt, Ht, Wt = target_shape

    if pad_value is None:
        pad_value = float(np.percentile(vol, 95))  # positive outside

    out = np.full(target_shape, pad_value, dtype=vol.dtype)

    def compute_slices(s, t):
        if s >= t:
            src0 = (s - t) // 2
            dst0 = 0
            length = t
        else:
            src0 = 0
            dst0 = (t - s) // 2
            length = s
        return src0, dst0, length

    sd0, dd0, ld = compute_slices(D, Dt)
    sh0, dh0, lh = compute_slices(H, Ht)
    sw0, dw0, lw = compute_slices(W, Wt)

    out[dd0:dd0+ld, dh0:dh0+lh, dw0:dw0+lw] = vol[sd0:sd0+ld, sh0:sh0+lh, sw0:sw0+lw]
    return out


In [4]:
def create_axis_aligned_loop_observation(
    sdf: np.ndarray,
    axis: str = "z",
    n_slices: int = 30,
    loop_band: float = 1.0,
    margin: int = 5,
):
    D, H, W = sdf.shape
    partial = np.zeros_like(sdf, dtype=np.float32)
    mask = np.zeros_like(sdf, dtype=np.float32)

    if axis == "z":
        idxs = np.linspace(margin, W - 1 - margin, n_slices).astype(int)
        for z in idxs:
            loop = (np.abs(sdf[:, :, z]) <= loop_band)
            partial[:, :, z][loop] = sdf[:, :, z][loop]
            mask[:, :, z][loop] = 1.0
        info = {"axis": "z", "slice_indices": idxs.tolist()}

    elif axis == "y":
        idxs = np.linspace(margin, H - 1 - margin, n_slices).astype(int)
        for y in idxs:
            loop = (np.abs(sdf[:, y, :]) <= loop_band)
            partial[:, y, :][loop] = sdf[:, y, :][loop]
            mask[:, y, :][loop] = 1.0
        info = {"axis": "y", "slice_indices": idxs.tolist()}

    elif axis == "x":
        idxs = np.linspace(margin, D - 1 - margin, n_slices).astype(int)
        for x in idxs:
            loop = (np.abs(sdf[x, :, :]) <= loop_band)
            partial[x, :, :][loop] = sdf[x, :, :][loop]
            mask[x, :, :][loop] = 1.0
        info = {"axis": "x", "slice_indices": idxs.tolist()}

    else:
        raise ValueError("axis must be 'x', 'y', or 'z'")

    return partial, mask, info


In [5]:
class UNetLoopsDataset(Dataset):
    def __init__(
        self,
        npz_paths,
        target_shape=(128,128,128),
        axis="z",
        n_slices=30,
        loop_band=1.0,
        seed=0
    ):
        self.axis = axis
        self.n_slices = n_slices
        self.loop_band = loop_band
        self.rng = np.random.default_rng(seed)

        self.sdfs = []
        self.names = []
        for p in npz_paths:
            sdf = np.load(p)["sdf"].astype(np.float32)
            sdf = pad_crop_to_shape_3d(sdf, target_shape=target_shape)
            self.sdfs.append(sdf)
            self.names.append(os.path.basename(p))

    def __len__(self):
        return len(self.sdfs)

    # En tu clase UNetLoopsDataset

    def __getitem__(self, idx):
        sdf = self.sdfs[idx]
        name = self.names[idx]

    # --- PASO NUEVO: Generar input parcial ---
        partial, mask, info = create_axis_aligned_loop_observation(
            sdf,
            axis=self.axis,
            n_slices=self.n_slices,
            loop_band=self.loop_band,
            margin=5
        )

    # --- PASO CRITICO: Normalización del Target (SDF) ---
    # 1. Definir una distancia de truncado (ej. 10 voxels). 
    # Todo lo que esté más lejos de 10 o -10 no nos importa tanto para la forma.
        trunc_dist = 10.0 
    
    # 2. Recortar (Clamp)
        sdf_clamped = np.clip(sdf, -trunc_dist, trunc_dist)
    
    # 3. Escalar a [-1, 1]
        sdf_normalized = sdf_clamped / trunc_dist

    # Preparar tensores
        x = np.stack([partial, mask], axis=0).astype(np.float32)
    # Usamos el SDF normalizado como target
        y = sdf_normalized[None].astype(np.float32)

        meta = {"name": name, "loop_info": info}
        return torch.from_numpy(x), torch.from_numpy(y), meta


In [6]:
def build_graph_from_loops(loops, k_inter=3):
    X_list = []
    slice_id_list = []
    offsets = []
    cur = 0

    for L in loops:
        pts = L["points"]
        offsets.append(cur)
        cur += len(pts)
        X_list.append(pts)
        slice_id_list.append(np.full(len(pts), L["slice_index"], dtype=np.int32))

    if len(X_list) == 0:
        return np.zeros((0,3), np.float32), np.zeros((0,2), np.int64), np.zeros((0,), np.int32)

    X = np.concatenate(X_list, axis=0).astype(np.float32)
    slice_id = np.concatenate(slice_id_list, axis=0)

    edges = []

    # Intra-loop (cyclic)
    for off, L in zip(offsets, loops):
        n = len(L["points"])
        if n < 2:
            continue
        for i in range(n):
            a = off + i
            b = off + ((i + 1) % n)
            edges.append((a, b))
            edges.append((b, a))

    # Inter-slice (adjacent slices)
    unique_slices = sorted(set([L["slice_index"] for L in loops]))
    slice_to_nodes = {}

    node_cursor = 0
    for L in loops:
        n = len(L["points"])
        s = L["slice_index"]
        slice_to_nodes.setdefault(s, []).extend(range(node_cursor, node_cursor + n))
        node_cursor += n

    for s in unique_slices:
        for s2 in (s - 1, s + 1):
            if s2 not in slice_to_nodes:
                continue
            A = np.array(slice_to_nodes[s], dtype=np.int32)
            B = np.array(slice_to_nodes[s2], dtype=np.int32)
            if len(A) == 0 or len(B) == 0:
                continue

            nbrs = NearestNeighbors(n_neighbors=min(k_inter, len(B))).fit(X[B])
            _, idx = nbrs.kneighbors(X[A])

            for ai, neighs in enumerate(idx):
                a = int(A[ai])
                for nb in neighs:
                    b = int(B[int(nb)])
                    edges.append((a, b))
                    edges.append((b, a))

    E = np.array(edges, dtype=np.int64) if len(edges) > 0 else np.zeros((0,2), np.int64)
    return X, E, slice_id


In [7]:
# 1) Convert meshes → npz
npz_paths = convert_all_meshes_to_sdf("meshes", "sdfs_npz", resolution=128, padding=5)

# 2) Create datasets
unet_dataset = UNetLoopsDataset(npz_paths, axis="z", n_slices=40, loop_band=1.0)

# 3) DataLoaders
unet_loader = DataLoader(unet_dataset, batch_size=1, shuffle=True)  # meta is dict -> ok

# 4) Quick sanity checks
x, gt, meta = unet_dataset[0]
print("UNet sample:", x.shape, gt.shape, meta)


Saved SDF to sdfs_npz\bunny_sdf_128.npz - shape: (120, 119, 93)
Saved SDF to sdfs_npz\cow_sdf_128.npz - shape: (39, 73, 120)
Saved SDF to sdfs_npz\mesh_133568_sdf_128.npz - shape: (119, 84, 111)
Saved SDF to sdfs_npz\mesh_386841_sdf_128.npz - shape: (86, 62, 120)
Saved SDF to sdfs_npz\mesh_48013_sdf_128.npz - shape: (114, 120, 64)
UNet sample: torch.Size([2, 128, 128, 128]) torch.Size([1, 128, 128, 128]) {'name': 'bunny_sdf_128.npz', 'loop_info': {'axis': 'z', 'slice_indices': [5, 8, 11, 14, 17, 20, 23, 26, 29, 32, 35, 38, 41, 44, 47, 50, 53, 56, 59, 62, 65, 68, 71, 74, 77, 80, 83, 86, 89, 92, 95, 98, 101, 104, 107, 110, 113, 116, 119, 122]}}


In [8]:
def visualize_unet_sample_3d(dataset, idx=0, level=0.0, max_points=30000):
    """
    Visualize one UNetLoopsDataset sample:
      - GT surface (marching cubes)
      - Loop voxels (mask==1)

    Args:
      dataset: UNetLoopsDataset
      idx: sample index
      level: SDF iso-level (usually 0)
      max_points: subsample loop voxels for visualization
    """
    x, y, meta = dataset[idx]
    sdf = y[0].numpy()
    mask = x[1].numpy()

    # Extract GT surface
    verts, faces, _, _ = measure.marching_cubes(sdf, level=level)

    # Loop voxels
    pts = np.argwhere(mask > 0.5)
    if len(pts) > max_points:
        sel = np.random.choice(len(pts), size=max_points, replace=False)
        pts = pts[sel]

    fig = go.Figure()

    # GT surface
    fig.add_trace(go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        opacity=0.25,
        color="lightblue",
        name="GT surface"
    ))

    # Loop voxels
    fig.add_trace(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode="markers",
        marker=dict(size=2, color="black"),
        name="Loop voxels"
    ))

    fig.update_layout(
        title=f"UNet sample | {meta['name']} | {meta['loop_info']}",
        scene=dict(aspectmode="data"),
        margin=dict(l=0, r=0, t=50, b=0)
    )
    fig.show()


In [40]:
visualize_unet_sample_3d(unet_dataset, idx=1)



In [9]:
class DoubleConv3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

def match_size(src, ref):
    # Center crop or pad src so its (D,H,W) matches ref
    sd, sh, sw = src.shape[-3:]
    rd, rh, rw = ref.shape[-3:]

    d0 = max((sd - rd) // 2, 0)
    h0 = max((sh - rh) // 2, 0)
    w0 = max((sw - rw) // 2, 0)
    src = src[:, :, d0:d0+rd, h0:h0+rh, w0:w0+rw]

    sd, sh, sw = src.shape[-3:]
    pd = max(rd - sd, 0)
    ph = max(rh - sh, 0)
    pw = max(rw - sw, 0)
    if pd or ph or pw:
        src = F.pad(src, (pw//2, pw - pw//2, ph//2, ph - ph//2, pd//2, pd - pd//2))
    return src

class UNet3DSmallRobust(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, base=16):
        super().__init__()
        self.down1 = DoubleConv3D(in_ch, base)
        self.pool1 = nn.MaxPool3d(2)
        self.down2 = DoubleConv3D(base, base*2)
        self.pool2 = nn.MaxPool3d(2)
        self.mid   = DoubleConv3D(base*2, base*4)

        self.up2   = nn.ConvTranspose3d(base*4, base*2, 2, stride=2)
        self.dec2  = DoubleConv3D(base*4, base*2)
        self.up1   = nn.ConvTranspose3d(base*2, base, 2, stride=2)
        self.dec1  = DoubleConv3D(base*2, base)

        self.out   = nn.Conv3d(base, out_ch, 1)

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(self.pool1(x1))
        xm = self.mid(self.pool2(x2))

        x = self.up2(xm)
        x = match_size(x, x2)
        x = self.dec2(torch.cat([x, x2], dim=1))

        x = self.up1(x)
        x = match_size(x, x1)
        x = self.dec1(torch.cat([x, x1], dim=1))

        return self.out(x)


In [10]:
def gradient_3d_forward(f):
    dx = f[:, :, 1:, :, :] - f[:, :, :-1, :, :]
    dy = f[:, :, :, 1:, :] - f[:, :, :, :-1, :]
    dz = f[:, :, :, :, 1:] - f[:, :, :, :, :-1]
    return dx, dy, dz

def eikonal_loss_3d(f):
    dx, dy, dz = gradient_3d_forward(f)
    dx_c = dx[:, :, :, :-1, :-1]
    dy_c = dy[:, :, :-1, :, :-1]
    dz_c = dz[:, :, :-1, :-1, :]
    grad_mag = torch.sqrt(dx_c**2 + dy_c**2 + dz_c**2 + 1e-6)
    return torch.mean(torch.abs(grad_mag - 1.0))

def laplacian_loss_3d(f):
    lap = (
        -6.0 * f
        + torch.roll(f,  1, dims=2) + torch.roll(f, -1, dims=2)
        + torch.roll(f,  1, dims=3) + torch.roll(f, -1, dims=3)
        + torch.roll(f,  1, dims=4) + torch.roll(f, -1, dims=4)
    )
    return torch.mean(lap**2)

def curvature_loss_3d(f):
    dxx = f[:, :, :, :, :-2] - 2*f[:, :, :, :, 1:-1] + f[:, :, :, :, 2:]
    dyy = f[:, :, :, :-2, :] - 2*f[:, :, :, 1:-1, :] + f[:, :, :, 2:, :]
    dzz = f[:, :, :-2, :, :] - 2*f[:, :, 1:-1, :, :] + f[:, :, 2:, :, :]
    return torch.mean(dxx**2) + torch.mean(dyy**2) + torch.mean(dzz**2)

def loss_function_3d(
    pred, gt, partial, mask,
    trunc_dist=10.0,
    λ_rec=1.0,
    λ_mask=200.0,
    λ_eik=0.01,
    λ_smooth=0.01,
    λ_curv=0.01
):
    # --- 1. PÉRDIDA DE RECONSTRUCCIÓN PONDERADA ---
    # Calculamos la diferencia absoluta
    diff = torch.abs(pred - gt)
    
    # Creamos un mapa de pesos: 
    # Todo vale 1.0 por defecto, pero el interior (gt < 0) vale 10.0
    weights = torch.ones_like(gt)
    weights[gt < 0] = 10.0 
    
    # Aplicamos los pesos a la diferencia
    L_rec = torch.mean(diff * weights)
    
    # --- 2. RESTO DE PÉRDIDAS (Igual que antes) ---
    L_mask = torch.mean(torch.abs((pred - partial) * mask))

    # Escalar a unidades reales para las pérdidas físicas
    pred_real_scale = pred * trunc_dist 

    L_eik = eikonal_loss_3d(pred_real_scale)
    L_smooth = laplacian_loss_3d(pred_real_scale)
    L_curv = curvature_loss_3d(pred_real_scale)

    return (
        λ_rec * L_rec
        + λ_mask * L_mask
        + λ_eik * L_eik
        + λ_smooth * L_smooth
        + λ_curv * L_curv
    )

In [11]:
def loss_function_with_sampling(
    pred, gt, partial, mask,
    trunc_dist=10.0,
    λ_rec=1.0,
    λ_mask=200.0,
    λ_eik=0.1,    # Aumentado para mejorar "manifoldness"
    λ_smooth=0.05, # Aumentado para "smoothing"
    num_samples=50000 # Cantidad de puntos a muestrear por batch
):
    # --- 1. ESTRATEGIA DE SAMPLING (LO QUE PIDIÓ EL PROFE) ---
    # No queremos entrenar con los 2 millones de voxels, elegimos unos pocos inteligentes.
    
    # A) Puntos cerca de la superficie (SDF real cercano a 0)
    # Convertimos a escala de pixels para filtrar: |gt| < 3 voxels de distancia
    surface_mask = torch.abs(gt * trunc_dist) < 3.0 
    
    # B) Puntos aleatorios en todo el volumen (para no olvidar el espacio vacío)
    random_mask = torch.rand_like(gt) > 0.95  # 5% de los puntos al azar
    
    # C) Combinar máscaras: Entrenamos en superficie O en puntos aleatorios
    training_mask = torch.logical_or(surface_mask, random_mask)
    
    # Si por mala suerte no hay puntos (batch vacío), usamos todo (fallback)
    if training_mask.sum() == 0:
        training_mask = torch.ones_like(gt, dtype=torch.bool)

    # --- 2. CÁLCULO DE PÉRDIDAS ---
    
    # Loss de Reconstrucción (SOLO en los puntos muestreados)
    # Esto responde a "Random samples of SDFs"
    diff = torch.abs(pred - gt)
    L_rec = torch.mean(diff[training_mask]) 

    # Loss de consistencia de datos (Input slices)
    # Aquí sí nos importa todo el input dado
    L_mask = torch.mean(torch.abs((pred - partial) * mask))

    # --- 3. REGULARIZACIÓN (Manifoldness & Smoothing) ---
    # El profe pidió "constraints". Calculamos esto en todo el volumen 
    # para asegurar que la física se cumpla en todas partes.
    
    pred_real_scale = pred * trunc_dist 
    
    # Eikonal (gradiente = 1)
    L_eik = eikonal_loss_3d(pred_real_scale)
    
    # Laplacian (suavizado)
    L_smooth = laplacian_loss_3d(pred_real_scale)
    
    return (
        λ_rec * L_rec
        + λ_mask * L_mask
        + λ_eik * L_eik      # Force manifoldness
        + λ_smooth * L_smooth # Force smoothing
    )

In [13]:

def train_unet(model, dataset, device="cuda", epochs=5, batch_size=1, lr=1e-3):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    model.to(device)
    model.train()

    for ep in range(1, epochs+1):
        total = 0.0
        for x, gt, meta in loader:
            x = x.to(device)      # (B,2,D,H,W)
            gt = gt.to(device)    # (B,1,D,H,W)

            pred = model(x)

            partial = x[:, 0:1]
            mask    = x[:, 1:2]

            loss = loss_function_with_sampling(pred, gt, partial, mask)

            opt.zero_grad()
            loss.backward()
            opt.step()

            total += float(loss.item())

        print(f"Epoch {ep}/{epochs} | loss={total/len(loader):.4f}")


In [14]:
def safe_marching_cubes(vol, preferred_level=0.0):
    vmin, vmax = float(vol.min()), float(vol.max())
    
    # FIX: We want to execute marching cubes ONLY if the level is INSIDE the range.
    # The previous logic (vmin <= level <= vmax: return None) was incorrect.
    if not (vmin < preferred_level < vmax):
        # Level is out of bounds, cannot extract surface
        return None, None, preferred_level, (vmin, vmax)
        
    verts, faces, _, _ = measure.marching_cubes(vol, level=preferred_level)
    return verts, faces, preferred_level, (vmin, vmax)

def visualize_prediction_vs_gt_safe(model, dataset, idx=0, device="cpu", shift=160, trunc_dist=10.0):
    model.eval()
    with torch.no_grad():
        x, gt, meta = dataset[idx]
        
        # Inference
        pred = model(x.unsqueeze(0).to(device)).cpu()[0,0].numpy()
        gt_np = gt[0].numpy()
        
        # --- DENORMALIZATION ---
        # Scale back to real units for visualization/analysis
        pred_real = pred * trunc_dist
        gt_real = gt_np * trunc_dist

    # Extract meshes
    v_gt, f_gt, lvl_gt, rg_gt = safe_marching_cubes(gt_real, preferred_level=0.0)
    v_pr, f_pr, lvl_pr, rg_pr = safe_marching_cubes(pred_real, preferred_level=0.0)

    # Setup Plotly figure
    fig = go.Figure()

    # Plot Ground Truth
    if v_gt is not None:
        fig.add_trace(go.Mesh3d(
            x=v_gt[:,0], y=v_gt[:,1], z=v_gt[:,2],
            i=f_gt[:,0], j=f_gt[:,1], k=f_gt[:,2],
            opacity=0.5, name=f"GT", color='blue'
        ))
    
    # Plot Prediction
    if v_pr is not None:
        v_pr_shifted = v_pr.copy()
        v_pr_shifted[:, 0] += shift  # Shift along X axis to show side-by-side
        
        fig.add_trace(go.Mesh3d(
            x=v_pr_shifted[:,0], y=v_pr_shifted[:,1], z=v_pr_shifted[:,2],
            i=f_pr[:,0], j=f_pr[:,1], k=f_pr[:,2],
            opacity=0.5, name=f"Pred", color='red'
        ))
    else:
        print(f"Warning: No surface found in prediction. Range: {rg_pr}")

    fig.update_layout(
        title=(f"{meta['name']} | {meta['loop_info']}<br>"
               f"GT Range: {rg_gt} | Pred Range: {rg_pr}"),
        scene=dict(aspectmode="data"),
        margin=dict(l=0, r=0, t=60, b=0)
    )
    fig.show()

In [18]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = UNet3DSmallRobust(in_ch=2, out_ch=1, base=16)

train_unet(model, unet_dataset, device=device, epochs=15, batch_size=1, lr=1e-3)

Epoch 1/15 | loss=1.7079
Epoch 2/15 | loss=1.6395
Epoch 3/15 | loss=1.5207
Epoch 4/15 | loss=1.3449
Epoch 5/15 | loss=1.1493
Epoch 6/15 | loss=1.0461
Epoch 7/15 | loss=0.9678
Epoch 8/15 | loss=0.9621
Epoch 9/15 | loss=1.0955
Epoch 10/15 | loss=0.9942
Epoch 11/15 | loss=1.0053
Epoch 12/15 | loss=0.9454
Epoch 13/15 | loss=0.8943
Epoch 14/15 | loss=0.8836
Epoch 15/15 | loss=0.8643


In [17]:
visualize_prediction_vs_gt_safe(model, unet_dataset, idx=1, device=device)

In [19]:
visualize_prediction_vs_gt_safe(model, unet_dataset, idx=1, device=device)